# RAG Experiments\n
\n
This notebook inspects the corpus, retrieval results, source-grounded answers and evaluation experiments. It imports reusable code from `src/rag_assistant` rather than duplicating the implementation.

## Environment\n
\n
Run the project installation command from the README before executing this notebook. Cells will be added as each pipeline component is implemented.

## 1. Corpus preparation

A RAG system needs an evidence corpus before it can answer questions. This project uses an explicit allowlist of official scikit-learn and pandas documentation pages. It does not crawl arbitrary web pages.

```text
Configured source URLs
        ↓
Downloaded raw HTML pages
        ↓
Provenance manifest with checksums
        ↓
Extracted heading-aware sections
        ↓
Chunks for retrieval (next step)
```

Raw HTML is the page content sent by a documentation website. It is stored locally so that extraction can be reproduced. Navigation, sidebars, scripts and other webpage elements are removed during extraction.

In [1]:
from collections import Counter
from pathlib import Path

from rag_assistant.config import load_sources

project_root = Path.cwd().resolve()
if not (project_root / 'pyproject.toml').exists():
    project_root = project_root.parent
if not (project_root / 'pyproject.toml').exists():
    raise RuntimeError('Open this notebook from the repository root or notebooks directory.')

sources = load_sources(project_root / 'config' / 'sources.toml')
print(f'Configured sources: {len(sources)}')
print(dict(Counter(source.library for source in sources)))
for source in sources[:5]:
    print(f'- {source.identifier}: {source.title}')

Configured sources: 25
{'scikit-learn': 17, 'pandas': 8}
- skl-pitfalls: Common pitfalls and recommended practices
- skl-compose: Pipelines and composite estimators
- skl-preprocessing: Preprocessing data
- skl-impute: Imputation of missing values
- skl-validation: Cross-validation: evaluating estimator performance


### Raw documents and provenance

`rag-assistant ingest` downloads every allowlisted page and saves the original HTML under `data/raw/`. It also creates `manifest.json`. The manifest records the source URL, retrieval time, content type, file size and SHA-256 checksum. A checksum is a fingerprint: when the downloaded page changes, its checksum changes too.

The raw files and manifest remain local and are excluded from Git.

In [2]:
import hashlib
import json

raw_directory = project_root / 'data' / 'raw'
manifest_path = raw_directory / 'manifest.json'
if not manifest_path.exists():
    raise FileNotFoundError('Run `rag-assistant ingest` from the repository root first.')

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
first_document = manifest['sources'][0]
print(f"Downloaded documents: {len(manifest['sources'])}")
print(f"Example document: {first_document['title']}")
print(f"Source URL: {first_document['retrieved_url']}")
print(f"Retrieved at: {first_document['retrieved_at_utc']}")
print(f"SHA-256: {first_document['sha256']}")

Downloaded documents: 25
Example document: Common pitfalls and recommended practices
Source URL: https://scikit-learn.org/stable/common_pitfalls.html
Retrieved at: 2026-09-30T09:57:46.268187+00:00
SHA-256: 21fac11c181ab35624dcf8673f0475d403d4f6ecf0f9bead01a573d7642b515c


### Extracted sections

`rag-assistant extract` reads the raw HTML and keeps the documentation article only. It preserves headings, paragraphs, list items and code blocks.

A **section** is a meaningful source unit under a heading. For example, a section may contain all the text under `Preprocessing data > Standardization`. A section is not yet a retrieval chunk: it can be too long for embedding and search. The next stage will split each section into smaller overlapping chunks while keeping its source metadata.

In [3]:
sections_path = project_root / 'data' / 'processed' / 'sections.jsonl'
if not sections_path.exists():
    raise FileNotFoundError('Run `rag-assistant extract` from the repository root first.')

sections = [json.loads(line) for line in sections_path.read_text(encoding='utf-8').splitlines()]
print(f'Extracted sections: {len(sections)}')
print(dict(Counter(section['library'] for section in sections)))

example_section = sections[1]
print(f"\nSection ID: {example_section['section_id']}")
print(f"Heading path: {' > '.join(example_section['section_path'])}")
print(f"Source: {example_section['source_url']}")
print('\nText preview:')
print(example_section['text'][:700])

Extracted sections: 645
{'scikit-learn': 337, 'pandas': 308}

Section ID: skl-pitfalls:002
Heading path: 12. Common pitfalls and recommended practices > 12.1. Inconsistent preprocessing
Source: https://scikit-learn.org/stable/common_pitfalls.html

Text preview:
scikit-learn provides a library of Dataset transformations , which
may clean (see Preprocessing data ), reduce
(see Unsupervised dimensionality reduction ), expand (see Kernel Approximation )
or generate (see Feature extraction ) feature representations.
If these data transforms are used when training a model, they also
must be used on subsequent datasets, whether it’s test data or
data in a production system. Otherwise, the feature space will change,
and the model will not be able to perform effectively.

For the following example, let’s create a synthetic dataset with a
single feature:

```
>>> from sklearn.datasets import make_regression >>> from sklearn.model_selection import train_test_


### Validation and next step

The initial corpus contains 25 documents and 645 non-empty extracted sections. Every raw file is checked against the SHA-256 value in the manifest, and every configured document is represented by extracted sections.

Next, we will inspect baseline chunking before embeddings are generated.

## 2. Baseline chunking

A source section can be too long to embed and retrieve precisely. The baseline splits each section into windows of 250 whitespace-separated units with 40 units of overlap. The overlap repeats the end of one chunk at the beginning of the next chunk, which keeps context around a boundary.

Each chunk keeps its section ID, document ID, heading path and source URL. This is a simple reference strategy; later it will be compared with structure-aware and model-tokenizer-based chunking.

In [ ]:
from rag_assistant.chunking import chunk_sections
from rag_assistant.config import load_chunking_settings

settings = load_chunking_settings(project_root / 'config' / 'settings.toml')
chunks_path = project_root / 'data' / 'processed' / 'chunks.jsonl'

def report_progress(completed, total, section_id):
    if completed == 1 or completed == total or completed % 25 == 0:
        print(f'Processed {completed}/{total} sections; latest: {section_id}')

chunks = chunk_sections(
    sections_path,
    chunks_path,
    settings,
    progress_callback=report_progress,
)

print(f'\nChunk size: {settings.chunk_size} units')
print(f'Chunk overlap: {settings.chunk_overlap} units')
print(f'Created chunks: {len(chunks)}')
print(f'Largest chunk: {max(chunk["unit_count"] for chunk in chunks)} units')

In [ ]:
from rag_assistant.chunking import chunk_section

long_section = next(section for section in sections if len(section['text'].split()) > settings.chunk_size)
section_chunks = chunk_section(long_section, settings)

print(f"Section: {' > '.join(long_section['section_path'])}")
print(f"Section units: {len(long_section['text'].split())}")
print(f"Chunks created: {len(section_chunks)}")

for chunk in section_chunks[:2]:
    print(f"\n{chunk['chunk_id']} | units {chunk['unit_start']} to {chunk['unit_end']}")
    print(chunk['text'][:500])

## 3. Embeddings

An embedding model converts each chunk into a vector: a fixed-length list of numbers representing its meaning. Similar chunks should have vectors that are close together. The selected open-source model supports both English documentation and French questions.

The model requires `passage: ` before stored documentation chunks and `query: ` before future user questions. Vectors and their manifest are local artifacts and are excluded from Git.

In [ ]:
from rag_assistant.config import load_embedding_settings
from rag_assistant.embeddings import create_embeddings

embedding_settings = load_embedding_settings(project_root / 'config' / 'settings.toml')
embedding_directory = project_root / 'artifacts' / 'embeddings'
embedding_file = embedding_directory / 'chunk_embeddings.npz'
manifest_path = embedding_directory / 'manifest.json'

current_chunks_sha256 = hashlib.sha256(chunks_path.read_bytes()).hexdigest()
existing_manifest = (
    json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest_path.exists()
    else {}
)
if embedding_file.exists() and existing_manifest.get('input_sha256') == current_chunks_sha256:
    embedding_manifest = existing_manifest
    print('Using existing local embeddings.')
else:
    embedding_manifest = create_embeddings(
        chunks_path,
        embedding_directory,
        embedding_settings,
        batch_size=32,
        show_progress_bar=True,
    )

print(embedding_manifest)

## 4. FAISS vector search

FAISS stores the chunk embeddings in a searchable vector index. This project uses `IndexFlatIP`: an exact inner-product search. Because embeddings are normalized, inner-product ranking is equivalent to cosine-similarity ranking.

Building the index reuses the saved chunk embeddings. It does not generate embeddings again. A search creates only one temporary embedding for the user question, then returns the top five source chunks.

In [ ]:
from rag_assistant.retrieval import build_faiss_index, search_chunks
from rag_assistant.config import load_retrieval_settings

faiss_directory = project_root / 'artifacts' / 'faiss'
index_file = faiss_directory / 'chunks.index'
faiss_manifest_path = faiss_directory / 'manifest.json'

current_embeddings_sha256 = hashlib.sha256(embedding_file.read_bytes()).hexdigest()
existing_faiss_manifest = (
    json.loads(faiss_manifest_path.read_text(encoding='utf-8'))
    if faiss_manifest_path.exists()
    else {}
)
if index_file.exists() and existing_faiss_manifest.get('embedding_artifact_sha256') == current_embeddings_sha256:
    faiss_manifest = existing_faiss_manifest
    print('Using existing local FAISS index.')
else:
    faiss_manifest = build_faiss_index(embedding_file, faiss_directory)

print(faiss_manifest)

In [ ]:
question = 'Comment traiter les valeurs manquantes ?'
retrieval_settings = load_retrieval_settings(project_root / 'config' / 'settings.toml')

results = search_chunks(
    question,
    chunks_path,
    embedding_file,
    index_file,
    embedding_settings,
    retrieval_settings,
)

for rank, result in enumerate(results, start=1):
    print(f"{rank}. score={result['score']:.4f}")
    print(f"   heading={' > '.join(result['section_path'])}")
    print(f"   source={result['source_url']}")
    print(f"   preview={' '.join(result['text'].split())[:350]}\n")

## 5. Grounded generation with Ollama

Retrieval supplies evidence; it does not write the final answer. The local Ollama model receives only the selected passages, the user question and the reusable prompt from `prompts/answer.txt`. The answer must cite the numbered passages and abstain when the retrieved evidence is insufficient.

The first generation cell requires the Ollama application and the model configured in `config/settings.toml` (currently `qwen2.5:1.5b`).

In [ ]:
from rag_assistant.config import load_generation_settings
from rag_assistant.generation import generate_answer, translate_retrieval_query

generation_settings = load_generation_settings(project_root / 'config' / 'settings.toml')
prompt_path = project_root / 'prompts' / 'answer.txt'

retrieval_query = translate_retrieval_query(question, generation_settings)
retrieved = search_chunks(
    retrieval_query, chunks_path, embedding_file, index_file,
    embedding_settings, retrieval_settings,
)
answer = generate_answer(
    question, retrieved, generation_settings, prompt_path, retrieval_query
)
print(f'Retrieval query: {retrieval_query}')
print(answer)

## 6. Several questions

A single example can hide fragile behavior. The following small set exercises missing values, pipelines, cross-validation, a French question and a question outside the selected corpus. The answers are printed with their retrieved source labels so that the result can be reviewed manually.

In [ ]:
questions = [
    'How can missing values be imputed in scikit-learn?',
    'What is the purpose of a Pipeline?',
    'Comment évaluer les performances d’un estimateur ?',
    'How should I deploy this project to Kubernetes?',
]

for question_item in questions:
    query_item = translate_retrieval_query(question_item, generation_settings)
    retrieved_item = search_chunks(
        query_item, chunks_path, embedding_file, index_file,
        embedding_settings, retrieval_settings,
    )
    answer_item = generate_answer(
        question_item, retrieved_item, generation_settings, prompt_path, query_item
    )
    print(f'\n### {question_item}\n{answer_item}')


## 7. Evaluation

The versioned evaluation set contains 15 questions: English and French queries, single-source and multi-source questions, and two questions deliberately outside the corpus. The evaluator checks whether an expected document was retrieved, whether citations refer to retrieved passages, and whether out-of-corpus questions trigger abstention. These automatic checks are useful regression signals, but they do not establish factual correctness.

In [ ]:
from rag_assistant.embeddings import load_embedding_model
from rag_assistant.evaluation import evaluate_questions
from rag_assistant.retrieval import load_retrieval_data

evaluation_output = project_root / 'artifacts' / 'evaluation'
retrieval_data = load_retrieval_data(chunks_path, embedding_file, index_file)
embedding_model = load_embedding_model(embedding_settings)
evaluation_results = evaluate_questions(
    questions_file=project_root / 'data' / 'evaluation' / 'questions.jsonl',
    output_directory=evaluation_output,
    chunks_file=chunks_path,
    embedding_file=embedding_file,
    index_file=index_file,
    prompt_file=prompt_path,
    embedding_settings=embedding_settings,
    retrieval_settings=retrieval_settings,
    generation_settings=generation_settings,
    embedding_model=embedding_model,
    retrieval_data=retrieval_data,
)

retrieval_passes = sum(item['retrieval_pass'] for item in evaluation_results)
answer_passes = sum(item['answer_check_pass'] for item in evaluation_results)
print(f'Retrieval checks passed: {retrieval_passes}/{len(evaluation_results)}')
print(f'Answer format checks passed: {answer_passes}/{len(evaluation_results)}')
print(f'Report: {evaluation_output / "evaluation_report.md"}')

## 8. Evaluation and model limits

This notebook demonstrates a transparent educational baseline, not a production-grade assistant. The allowlist covers only 25 documentation pages, so correct answers outside that corpus cannot be expected. Fixed-window chunks can split a concept across boundaries, and dense retrieval may miss exact terminology.

The small local generation model can confuse related concepts, omit constraints or produce plausible wording that is not fully supported by its passages. Citation labels show which passages were supplied, but they do not prove that every sentence is entailed by those passages. The automatic evaluation therefore needs manual review, especially for factual accuracy, completeness, language quality and safe abstention.

Useful follow-up experiments include tokenizer-aware or structure-aware chunking, hybrid lexical plus dense retrieval, reranking, a stronger local model, and a larger human-reviewed evaluation set.